In [ ]:
import pandas as pd
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.config import MERGED_DATA
def statement_to_dict_df(file, statement_name):
    ticker = Path(file).stem.replace(f"_{statement_name}", "")

    df = pd.read_csv(file)

    rows = []

    # First column contains metric names
    metric_col = df.columns[0]

    # Remaining columns are dates
    for date in df.columns[1:]:
        metrics = (
            df[[metric_col, date]]
            .dropna()
            .set_index(metric_col)[date]
            .to_dict()
        )

        rows.append({
            "company": ticker,
            "date": date,
            statement_name: metrics
        })

    return pd.DataFrame(rows)

In [3]:
from pathlib import Path

DATA_DIR = Path("../data")
BALANCE_SHEET_DIR = DATA_DIR / "balance_sheet"
CASH_FLOW_DIR = DATA_DIR / "cashflows"
FINANCIALS_DIR = DATA_DIR / "financials"
DATA_TICKER = DATA_DIR / "processed" / "layoffs_with_tickers.csv"
tickers = set()

for file in BALANCE_SHEET_DIR.glob("*.csv"):
    ticker = file.stem.replace("_balancesheet", "")
    tickers.add(ticker)
 
print(f"Found {len(tickers)} companies")

Found 2708 companies


In [5]:
master_rows = []

for ticker in tickers:

    dfs = []

    statement_info = [
        ("balancesheet", BALANCE_SHEET_DIR, "_balancesheet.csv"),
        ("cashflow", CASH_FLOW_DIR, "_cashflow.csv"),
        ("financials", FINANCIALS_DIR, "_financials.csv"),
    ]

    for statement_name, directory, suffix in statement_info:

        file = directory / f"{ticker}{suffix}"

        if not file.exists():
            continue

        try:
            statement_df = statement_to_dict_df(
                file,
                statement_name
            )

            if not statement_df.empty:
                dfs.append(statement_df)

        except Exception as e:
            print(
                f"Failed {statement_name} "
                f"for {ticker}: {e}"
            )

    # Skip companies with no statements
    if len(dfs) == 0:
        continue

    # Merge all available statements
    company_df = dfs[0]

    for df in dfs[1:]:
        company_df = company_df.merge(
            df,
            on=["company", "date"],
            how="outer"
        )

    master_rows.append(company_df)

    print(f"Processed {ticker}")

Processed AFCONS.BO
Processed DEO
Processed ASIX
Processed NRIS
Processed BIO
Processed AGRDF
Processed ARGL.CN
Processed FTI
Processed D8Y.F
Processed HRTX
Processed ACCO
Processed FSLY
Processed IOT
Processed ATKR
Processed LEE
Processed NKE
Processed PLUG
Processed 301559.SZ
Processed BAS.DE
Processed VOYA
Processed NOMD
Processed EC8.F
Processed FRO
Processed PATH
Processed EZPW
Processed NCH2.F
Processed LION
Processed GXO
Processed 3850.T
Processed PCSGH.BK
Processed PGIL.NS
Processed LCID
Processed RMNI
Processed CAI
Processed COCHW
Processed VASUPRADA.BO
Processed UNFI
Processed FIS
Processed KLN.MI
Processed WFG
Processed SNPS
Processed TSM
Processed NKTR
Processed 5236.KL
Processed HUBS
Processed TWST
Processed JBHT
Processed BTGO
Processed TOON
Processed EQ
Processed CRM
Processed CSN.SG
Processed WMMVY
Processed IBRX
Processed SW.PA
Processed 000180.KS
Processed COHN
Processed LYTS
Processed WOR
Processed IDR
Processed FCEL
Processed MO
Processed 0347.KL
Processed DKS
Proce

In [6]:
import pandas as pd

dataset = pd.concat(master_rows, ignore_index=True)
dataset["date"] = pd.to_datetime(dataset["date"])

dataset["quarter"] = dataset["date"].dt.to_period("Q").astype(str)

bs_features = pd.json_normalize(
    dataset["balancesheet"]
).add_prefix("bs_")


cf_features = pd.json_normalize(
    dataset["cashflow"]
).add_prefix("cf_")

fin_features = pd.json_normalize(
    dataset["financials"]
).add_prefix("fin_")

features_df = pd.concat(
    [
        dataset[["company", "date", "quarter"]],
        bs_features,
        cf_features,
        fin_features,
    ],
    axis=1,
)
# features_df.fillna("NaN")
# print(features_df.fillna("NaN").head())

features_df.to_csv(
    MERGED_DATA["MERGED_OUTPUT_CSV_PATH"],
    index=False
)

# Verify
print(features_df.shape)
print(features_df.head())

(12047, 349)
     company       date quarter  bs_Ordinary Shares Number  bs_Share Issued  \
0  AFCONS.BO 2024-09-30  2024Q3                        NaN              NaN   
1  AFCONS.BO 2024-12-31  2024Q4                        NaN              NaN   
2  AFCONS.BO 2025-03-31  2025Q1                367784631.0      367784631.0   
3  AFCONS.BO 2025-06-30  2025Q2                        NaN              NaN   
4  AFCONS.BO 2025-09-30  2025Q3                367784631.0      367784631.0   

    bs_Net Debt  bs_Total Debt  bs_Tangible Book Value  bs_Invested Capital  \
0           NaN            NaN                     NaN                  NaN   
1           NaN            NaN                     NaN                  NaN   
2  1.795550e+10   2.343300e+10            5.259830e+10         7.496240e+10   
3           NaN            NaN                     NaN                  NaN   
4  3.097260e+10   3.565230e+10            5.388340e+10         8.861110e+10   

   bs_Working Capital  ...  fin_Net I